In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = pd.read_csv("/content/sample_data/ds_salaries.csv")


In [ ]:
def group_job(title):
    title = str(title).lower().strip()

    # 1. Management / Leadership
    if "manager" in title or "director" in title or "head" in title:
        return "Management"

    # 2. Machine Learning / AI
    elif (
        "machine learning" in title
        or title == "ml engineer"
        or "ai" in title
        or "computer vision" in title
        or "nlp" in title
    ):
        return "ML_AI"

    # 3. Data Engineering
    elif (
        "data engineer" in title
        or "big data engineer" in title
        or "etl" in title
        or "architect" in title
        or "analytics engineer" in title
        or "data analytics engineer" in title
        or "cloud data engineer" in title
    ):
        return "Data_Engineering"

    # 4. Data Analysis
    elif "analyst" in title or "analytics lead" in title:
        return "Data_Analysis"

    # 5. Data Science
    elif "scientist" in title or "data science consultant" in title or "data science engineer" in title:
        return "Data_Science"

    # 6. Other
    else:
        return "Other"

df["job_group"] = df["job_title"].apply(group_job)
# =========================
# 3. DROP UNUSED COLUMNS
# =========================
columns_to_drop = ["Unnamed: 0", "salary", "salary_currency", "job_title", "employee_residence"]
df = df.drop(columns=columns_to_drop, errors="ignore")



In [ ]:
# =========================
# 4. GROUP COMPANY LOCATION
# =========================
top_8_locations = df["company_location"].value_counts().head(8).index.tolist()

df["company_location_grouped"] = df["company_location"].apply(
    lambda x: x if x in top_8_locations else "Other"
)

df = df.drop(columns=["company_location"])

In [ ]:

# =========================
# 5. ENCODE ORDINAL FEATURES
# =========================
experience_map = {
    "EN": 0,
    "MI": 1,
    "SE": 2,
    "EX": 3
}

size_map = {
    "S": 0,
    "M": 1,
    "L": 2
}

df["experience_level"] = df["experience_level"].map(experience_map)
df["company_size"] = df["company_size"].map(size_map)

In [ ]:
# =========================
# 6. LOG TRANSFORM TARGET
# =========================
df["salary_in_usd"] = np.log1p(df["salary_in_usd"])


# =========================
# 7. ONE-HOT ENCODE CATEGORICALS
# =========================
df = pd.get_dummies(
    df,
    columns=["employment_type", "company_location_grouped", "job_group"],
    drop_first=False,
    dtype=int
)

In [ ]:
# =========================
# 8. PREPARE X, y
# =========================
X = df.drop(columns=["salary_in_usd"])
y = df["salary_in_usd"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Columns:")
print(X.columns.tolist())


# =========================
# 9. TRAIN MODEL
# =========================
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

cv_results = cross_validate(
    estimator=rf_model,
    X=X,
    y=y,
    cv=kf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

results_df = pd.DataFrame({
    "Fold": range(1, 6),
    "Train MAE": -cv_results["train_mae"],
    "Validation MAE": -cv_results["test_mae"],
    "Train MSE": -cv_results["train_mse"],
    "Validation MSE": -cv_results["test_mse"],
    "Train RMSE": -cv_results["train_rmse"],
    "Validation RMSE": -cv_results["test_rmse"],
    "Train R2": cv_results["train_r2"],
    "Validation R2": cv_results["test_r2"],
})

print("\nCross-validation results:")
print(results_df)

print("\nAverage Cross-Validation Performance:")
print(f"Mean Train MAE:       {(-cv_results['train_mae']).mean():.2f}")
print(f"Mean Validation MAE:  {(-cv_results['test_mae']).mean():.2f}")
print(f"Mean Train MSE:       {(-cv_results['train_mse']).mean():.2f}")
print(f"Mean Validation MSE:  {(-cv_results['test_mse']).mean():.2f}")
print(f"Mean Train RMSE:      {(-cv_results['train_rmse']).mean():.2f}")
print(f"Mean Validation RMSE: {(-cv_results['test_rmse']).mean():.2f}")
print(f"Mean Train R2:        {cv_results['train_r2'].mean():.4f}")
print(f"Mean Validation R2:   {cv_results['test_r2'].mean():.4f}")


X shape: (607, 23)
y shape: (607,)
Columns:
['work_year', 'experience_level', 'remote_ratio', 'company_size', 'employment_type_CT', 'employment_type_FL', 'employment_type_FT', 'employment_type_PT', 'company_location_grouped_CA', 'company_location_grouped_DE', 'company_location_grouped_ES', 'company_location_grouped_FR', 'company_location_grouped_GB', 'company_location_grouped_GR', 'company_location_grouped_IN', 'company_location_grouped_Other', 'company_location_grouped_US', 'job_group_Data_Analysis', 'job_group_Data_Engineering', 'job_group_Data_Science', 'job_group_ML_AI', 'job_group_Management', 'job_group_Other']

Cross-validation results:
   Fold  Train MAE  Validation MAE  Train MSE  Validation MSE  Train RMSE  \
0     1   0.210268        0.408465   0.083488        0.376193    0.288944   
1     2   0.206086        0.404404   0.085571        0.319512    0.292525   
2     3   0.221939        0.322469   0.096455        0.199367    0.310573   
3     4   0.204085        0.380136   0.0

In [ ]:
# =========================
# 10. CROSS-VAL PREDICTIONS
# =========================
y_pred_cv = cross_val_predict(
    estimator=rf_model,
    X=X,
    y=y,
    cv=kf,
    n_jobs=-1
)

mae = mean_absolute_error(y, y_pred_cv)
mse = mean_squared_error(y, y_pred_cv)
rmse = np.sqrt(mse)
r2 = r2_score(y, y_pred_cv)

print("\nOverall 5-Fold CV Metrics:")
print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")



Overall 5-Fold CV Metrics:
MAE:  0.3854
MSE:  0.3066
RMSE: 0.5537
R2:   0.4875


In [ ]:
# =========================
# 11. FIT FINAL MODEL ON ALL DATA
# =========================
rf_model.fit(X, y)


# =========================
# 12. SAVE ARTIFACTS
# =========================
joblib.dump(rf_model, "salary_model.pkl")
joblib.dump(X.columns.tolist(), "model_columns.pkl")
joblib.dump(top_8_locations, "top_locations.pkl")
joblib.dump(experience_map, "experience_map.pkl")
joblib.dump(size_map, "size_map.pkl")

print("\nArtifacts saved successfully:")
print("- salary_model.pkl")
print("- model_columns.pkl")
print("- top_locations.pkl")
print("- experience_map.pkl")
print("- size_map.pkl")



Artifacts saved successfully:
- salary_model.pkl
- model_columns.pkl
- top_locations.pkl
- experience_map.pkl
- size_map.pkl


In [ ]:
# =========================
# 13. PREPROCESS FUNCTION FOR NEW INPUT
# =========================
def preprocess_input(input_dict, model_columns, top_locations, experience_map, size_map):
    df_input = pd.DataFrame([input_dict])

    # encode ordinal fields
    df_input["experience_level"] = df_input["experience_level"].map(experience_map)
    df_input["company_size"] = df_input["company_size"].map(size_map)

    # group job title
    df_input["job_group"] = df_input["job_title"].apply(group_job)
    df_input = df_input.drop(columns=["job_title"], errors="ignore")

    # group company location
    df_input["company_location_grouped"] = df_input["company_location"].apply(
        lambda x: x if x in top_locations else "Other"
    )
    df_input = df_input.drop(columns=["company_location"], errors="ignore")

    # drop employee_residence if sent by user
    df_input = df_input.drop(columns=["employee_residence"], errors="ignore")

    # one-hot encode
    df_input = pd.get_dummies(
        df_input,
        columns=["employment_type", "company_location_grouped", "job_group"],
        drop_first=False,
        dtype=int
    )

    # add missing columns
    for col in model_columns:
        if col not in df_input.columns:
            df_input[col] = 0

    # remove extra columns and reorder
    df_input = df_input[model_columns]

    return df_input

In [ ]:
# =========================
# 14. TEST ONE RAW INPUT
# =========================
sample_input = {
    "experience_level": "SE",
    "employment_type": "FT",
    "company_size": "M",
    "remote_ratio": 100,
    "company_location": "US",
    "employee_residence": "US",
    "job_title": "Data Scientist"
}

loaded_model = joblib.load("salary_model.pkl")
loaded_columns = joblib.load("model_columns.pkl")
loaded_top_locations = joblib.load("top_locations.pkl")
loaded_experience_map = joblib.load("experience_map.pkl")
loaded_size_map = joblib.load("size_map.pkl")

processed_sample = preprocess_input(
    input_dict=sample_input,
    model_columns=loaded_columns,
    top_locations=loaded_top_locations,
    experience_map=loaded_experience_map,
    size_map=loaded_size_map
)

pred_log = loaded_model.predict(processed_sample)
pred_salary = np.expm1(pred_log)[0]

print("\nProcessed sample:")
print(processed_sample)

print(f"\nPredicted salary (USD): {pred_salary:.2f}")


Processed sample:
   work_year  experience_level  remote_ratio  company_size  \
0          0                 2           100             1   

   employment_type_CT  employment_type_FL  employment_type_FT  \
0                   0                   0                   1   

   employment_type_PT  company_location_grouped_CA  \
0                   0                            0   

   company_location_grouped_DE  ...  company_location_grouped_GR  \
0                            0  ...                            0   

   company_location_grouped_IN  company_location_grouped_Other  \
0                            0                               0   

   company_location_grouped_US  job_group_Data_Analysis  \
0                            1                        0   

   job_group_Data_Engineering  job_group_Data_Science  job_group_ML_AI  \
0                           0                       1                0   

   job_group_Management  job_group_Other  
0                     0            

In [ ]:
df["company_location"].unique()

KeyError: 'company_location'